Objectif : générer un prompt de génération de vidéo à partir des transcriptions des audios

In [ ]:
import os
import time
from openai import OpenAI 
from dotenv import load_dotenv


# Charger l'api key de open ai
load_dotenv()
api_key = os.environ.get("OPENAI_KEY")

client = OpenAI(api_key=api_key)


def create_video_prompt(transcription_text, model="gpt-4o-mini"):
    system_prompt = (
        "Tu es un assistant créatif qui transforme un texte en prompt de vidéo mème courte. "
        "Le prompt doit inclure : style visuel, ambiance (mood), couleurs, personnages, actions, "
        "cadre (intérieur/extérieur), durée approximative, et 2-3 instructions techniques pour la génération comme la résolution de la vidéo en 720p,"
        "et le niveau de détail haut"
    )

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": transcription_text}
    ]

    # gpt-4o-mini ne supporte pas 'temperature'
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        max_completion_tokens= 1000 # <- Changement à 1000 tokens car necessisté de plus de détails, prompt pas complet à chaque fois avec 200 mots
    )

    return resp.choices[0].message.content.strip()


if __name__ == "__main__":
    input_folder = "../Transcriptions"
    output_folder = "../Prompts_generation_video"
    os.makedirs(output_folder, exist_ok=True)

    for filename in os.listdir(input_folder):
        if filename.endswith(".txt"):
            with open(os.path.join(input_folder, filename), "r", encoding="utf-8") as f:
                text = f.read().strip()

            try:
                prompt = create_video_prompt(text, model="gpt-4o-mini")
            except Exception as e:
                print("Erreur API :", e)
                time.sleep(1)
                continue

            outpath = os.path.join(output_folder, f"{os.path.splitext(filename)[0]}_prompt.txt")
            with open(outpath, "w", encoding="utf-8") as fw:
                fw.write(prompt)

            print(f"Prompt généré pour {filename}")


Prompt généré pour A one minute story|Short Stories|A one minute story in English#Shortstoriesenglish #oneminutestories.txt
Prompt généré pour Rabbit And Crow | One Minute Story | Childenzia English Story.txt
Prompt généré pour The Goose and Its Golden Egg | Moral Stories | Animated Stories.txt


# Génération de vidéo à partir de modèle Hugging face

La génération de vidéo se déroule en deux étapes :

        1. La fusion de tous les parties de prompt en un seul prompt

        2. L'utilisation de fal-ai/hunyuan-video comme model de génération text to vidéo (le mieux noté sur Hugging Face) 

            -> On utilise le modèle via l'API de la plateforme fal.ai spécifique de la génération de contenue, plus facile à mettre en place que sur Hugging Face.

In [ ]:
import fal_client
import os

# 1. Configuration de la sécurité
os.environ["FAL_KEY"] = "VOTRE_CLE_API_ICI"

def load_prompt_from_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as f:
        # .strip() enlève les espaces et retours à la ligne inutiles au début et à la fin
        return f.read().strip()


prompt_files = sorted([x for x in os.listdir("../Transcriptions/") if x.startswith("Rabbit")])
 

# Fusion de toutes les parties du prompt
parties = []
for filename in prompt_files:
    full_path = os.path.join("../Transcriptions/", filename)
    with open(full_path, 'r', encoding='utf-8') as f:
        parties.append(f.read().strip())

full_prompt = ",".join(parties)

load_dotenv()
api_key = os.environ.get("FAL_KEY")

def generate_video_demo(prompt_text):
    # 2. Définition du prompt

    print(f"🚀 Initialisation de la génération pour : {prompt_text}")

    # 3. Appel de l'API (Méthode Subscribe)
    handler = fal_client.subscribe(
        "fal-ai/hunyuan-video",
        arguments={
            "prompt": prompt_text,
            "video_size": "720p_portrait",
            "num_frames": 129,
            "fps": 24
        },
        with_logs=True,
        on_task_update=lambda update: print(f"⏳ État : {update.logs[-1]['message']}") if update.logs else None
    )

    # 4. Récupération du résultat
    result = handler.get()

    # 5. Affichage du lien
    print("\n✅ Génération terminée avec succès !")
    print(f"🔗 Lien de la vidéo : {result['video']['url']}")

if __name__ == "__main__":
    generate_video_demo(full_prompt)

Rabbit and Crow, a 1 minute story. Once upon a time there was a rabbit.,He always tried to copy other animals. One day he saw a crow. The crow was sitting on a tree.,He and doing nothing. Rabid thought to do the same. He sat on the ground and doing nothing when a fox.,Oh him. Fox came near and ate him. Model of the story is to sit and do not.,you need to be on top.
